# Bifrost.jl — visual demos

One notebook of runnable, human-inspected demos of the Bifrost API. Cells exercising
Bifrost are written to be read (duplication is deliberate); all plotting machinery
lives in [`demo-helper.jl`](demo-helper.jl) (`dh_*` functions). Each demo's lesson is
recorded alongside it and in [`demo-intent.md`](demo-intent.md).

**Running**: any Julia 1.11 kernel; the setup cell activates the environment pinned
by `test/human/Project.toml`. Plots are interactive Plotly figures rendered inline.

| § | Class |
| --- | --- |
| [1 · Smallest runnable examples](#sec1) | end-to-end propagation, deterministic and MCM |
| [2 · Path geometry](#sec2) | authoring, the 3D inspector, `:inherit`, connectors |
| [3 · Modify — field-level MCM meta](#sec3) | what each segment field means |
| [4 · Material spin](#sec4) | spin carried independently of frame geometry |
| [5 · Adaptive step-doubling](#sec5) | the solver's step controller at work |
| [6 · JumpBy / JumpTo — 2D sweeps](#sec6) | local-frame vs global-frame connectors |
| [7 · Meta × JumpTo interplay](#sec7) | anchored vs unanchored perturbations |
| [8 · JumpBy / JumpTo — 3D scenes](#sec8) | G2 continuity knobs and inheritance |
| [9 · MCM temperature PTF](#sec9) | ensemble uncertainty end to end |
| [10 · MCM speed benchmarks](#sec10) | ensemble cost vs Float64 |

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using Bifrost
using MonteCarloMeasurements
using Random
include("demo-helper.jl")
using .DemoHelper

<a id="sec1"></a>
## 1 · Smallest runnable examples

**Lesson**: the full layer stack — material → cross-section → geometry → fiber →
propagation — in a dozen lines. The 90° bend contributes bend birefringence with a
fixed axis, so `J` is diagonal, `diag(e^{-iφ}, e^{+iφ})`: a pure lossless retarder.
`stats` has 3 entries — the propagator never steps across segment boundaries.

In [ ]:
xs = StepIndexCrossSection(
    SilicaGermaniaGlass(0.036),
    SilicaGermaniaGlass(0.0),
    8.2e-6,
    125e-6;
    manufacturer = "Corning",
    model_number = "SMF-like",
)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5, meta = [Nickname("lead-in")])
bend!(sb;     radius = 0.05, angle = π / 2, meta = [Nickname("90° bend")])
straight!(sb; length = 0.5, meta = [Nickname("lead-out")])
seal!(sb)

fiber = Fiber(build(sb); cross_section = xs, T_ref_K = 297.15)
J, stats = propagate_fiber(fiber; λ_m = 1550e-9)

println("J =")
display(J)
println("intervals = ", length(stats))

### 1.1 · Smallest MCM example

**Lesson**: making one input uncertain is the only change needed for ensemble
propagation. Here the operating temperature is `T_ref_K = (297.15 ± 2.0) K`
(`Particles`); one `propagate_fiber` call propagates the whole ensemble and the
entries of `J` come back as `Particles` (displayed mean ± std). MCM blocks wrap in
`unsafe_comparisons(true)` per the package contract.

In [ ]:
Random.seed!(20260612)    # deterministic ensemble draws for a reproducible notebook
MonteCarloMeasurements.unsafe_comparisons(true)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5)
bend!(sb;     radius = 0.05, angle = π / 2)
straight!(sb; length = 0.5)
seal!(sb)

fiber = Fiber(build(sb); cross_section = xs, T_ref_K = 297.15 ± 2.0)
J_p, _ = propagate_fiber(fiber; λ_m = 1550e-9)
MonteCarloMeasurements.unsafe_comparisons(false)

display(J_p)

<a id="sec2"></a>
## 2 · Path geometry

The geometry layer alone — no optics. Authoring lifecycle on a `SubpathBuilder`:
`start!` → segment calls → seal (`jumpto!` to a global target or `seal!` at the
natural exit) → `build`.

Figures use the **path inspector** (`dh_path_inspector`): the slider scrubs arc
length, carrying the local T̂/N̂/B̂ triad, the normal–binormal plane, a red arrow at
the accumulated spin phase, and a live readout (s; x, y, z; κ; τ_geom; τ_spin;
∫τ_spin ds). Axes are fixed to the content's bounding box.

### 2.1 · A simple multi-segment Subpath

**Lesson**: the four curve primitives compose with tangent (G1) continuity at every
join, and the built path answers scalar queries — `path_length`, and `writhe`
(zero: this path is planar).

In [ ]:
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.10, meta = [Nickname("Straight")])
bend!(sb;     radius = 0.05, angle = π / 2, meta = [Nickname("Bend")])
straight!(sb; length = 0.12, meta = [Nickname("Straight")])
catenary!(sb; a = 0.03, length = 0.10, axis_angle = 0.0, meta = [Nickname("Catenary")])
bend!(sb;     radius = 0.06, angle = π / 3, meta = [Nickname("Bend")])
straight!(sb; length = 0.08, meta = [Nickname("Straight")])
seal!(sb)
path = build(sb)

println("Arc length: ", path_length(path), " m")
println("Writhe:     ", writhe(path; n = 128))
dh_path_inspector(path; title = "Single Subpath: straights, bends, catenary")

### 2.2 · Segment nicknames

**Lesson**: `Nickname` meta attaches at the segment call and flows through the build
to presentation — names live with the segment definition, never derived from type
information at render time. Use role names (`lead-in`, `sag`, …) where the role is
clearer than the type.

In [ ]:
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.08, meta = [Nickname("lead-in")])
bend!(sb;     radius = 0.06, angle = π / 2, meta = [Nickname("90° bend")])
straight!(sb; length = 0.06, meta = [Nickname("spacer")])
catenary!(sb; a = 0.04, length = 0.08, axis_angle = 0.0, meta = [Nickname("sag")])
helix!(sb;    radius = 0.025, pitch = 0.015, turns = 1.2, axis_angle = 0.0,
              meta = [Nickname("spin section")])
straight!(sb; length = 0.06, meta = [Nickname("lead-out")])
seal!(sb)
path = build(sb)

println("Arc length: ", path_length(path), " m")
dh_path_inspector(path; title = "Path geometry: segment nicknames")

### 2.3 · Helix `axis_angle`

**Lesson**: `axis_angle` rotates the helix axis about the incoming tangent. Entry
stays tangent-continuous for every value; the exit direction (and the lead-out
straight) swings around; arc length is invariant, as the printout confirms. The
helix segment is the subject (red) in the comparison row; the inspector below
scrubs the most out-of-plane variant.

In [ ]:
function helix_path(axis_angle)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 0.05, meta = [Nickname("lead-in")])
    helix!(sb;    radius = 0.03, pitch = 0.02, turns = 2.0, axis_angle = axis_angle,
                  meta = [Nickname("helix")])
    straight!(sb; length = 0.05, meta = [Nickname("lead-out")])
    seal!(sb)
    return build(sb)
end

for aa in (0.0, π / 3, 2π / 3)
    println("axis_angle = ", round(aa; digits = 4),
            ":  arc length = ", round(path_length(helix_path(aa)); digits = 4), " m")
end

dh_variant_row([("axis_angle = 0",    helix_path(0.0),    [2]),
                ("axis_angle = π/3",  helix_path(π / 3),  [2]),
                ("axis_angle = 2π/3", helix_path(2π / 3), [2])];
               spacing = 0.25,
               title = "HelixSegment axis_angle: same entry tangency, rotated axis")

In [ ]:
dh_path_inspector(helix_path(2π / 3); title = "HelixSegment: axis_angle = 2π/3")

### 2.4 · The paddle: five Subpaths, `:inherit`, `min_bend_radius`

**Lesson**: a `PathBuilt` shares one global arc length across sealed Subpaths;
`start!(sb, :inherit)` chains each start state from the previous endpoint; every
`jumpto!` seal takes its own `min_bend_radius`. The salient segment is Subpath 4's
*interior* `jumpby!`: its `delta` is local-frame, and the sealing `jumpto!` then
pins an exact global landing point.

In [ ]:
sb1 = SubpathBuilder(); start!(sb1)
straight!(sb1; length = 1.0, meta = [Nickname("Sub1 straight")])
jumpto!(sb1; point = (1.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.4)

sb2 = SubpathBuilder(); start!(sb2, :inherit)
straight!(sb2; length = 1.0, meta = [Nickname("Sub2 straight")])
jumpto!(sb2; point = (2.0, 0.0, 0.0), incoming_tangent = (0.0, 0.0, 1.0),
        min_bend_radius = 0.1)

sb3 = SubpathBuilder(); start!(sb3, :inherit)
straight!(sb3; length = 1.0, meta = [Nickname("Sub3 straight")])
jumpto!(sb3; point = (3.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.05)

sb4 = SubpathBuilder(); start!(sb4, :inherit)
straight!(sb4; length = 1.0, meta = [Nickname("Sub4 straight")])
jumpby!(sb4; delta = (-1.0, 0.0, 0.0), tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.1, meta = [Nickname("Sub4 JumpBy")])
jumpto!(sb4; point = (2.0, 0.0, 0.0), incoming_tangent = (0.0, 0.0, 1.0))

sb5 = SubpathBuilder(); start!(sb5, :inherit)
straight!(sb5; length = 1.0, meta = [Nickname("Sub5 straight")])
seal!(sb5)

paddle = build([sb1, sb2, sb3, sb4, sb5])
println("PathBuilt arc length: ", path_length(paddle), " m")
dh_path_inspector(paddle; fidelity = 4.0,
                  title = "JumpBy/JumpTo paddle: PathBuilt of 5 Subpaths")

### 2.5 · PathBuilt assembly with exact handoffs

**Lesson**: a `jumpto!` seal pins exact handoff coordinates and the build's
conformity check validates every Subpath boundary; per-Subpath `Nickname` meta
(passed to the `SubpathBuilder` constructor) labels whole subpaths.

In [ ]:
sb1 = SubpathBuilder(meta = [Nickname("Subpath 1: straight")])
start!(sb1)
straight!(sb1; length = 0.2, meta = [Nickname("Straight")])
jumpto!(sb1; point = (0.0, 0.0, 0.2), incoming_tangent = (0.0, 0.0, 1.0))

R = 0.05
sb2 = SubpathBuilder(meta = [Nickname("Subpath 2: bend")])
start!(sb2, :inherit)
bend!(sb2; radius = R, angle = π / 2, meta = [Nickname("90° bend")])
jumpto!(sb2; point = (R, 0.0, 0.2 + R), incoming_tangent = (1.0, 0.0, 0.0))

sb3 = SubpathBuilder(meta = [Nickname("Subpath 3: helix")])
start!(sb3, :inherit)
helix!(sb3; radius = 0.025, pitch = 0.02, turns = 1.5, axis_angle = 0.0,
       meta = [Nickname("Helix")])
seal!(sb3)

path = build([sb1, sb2, sb3])
println("Subpaths:   ", length(path.subpaths))
println("Arc length: ", path_length(path), " m")
dh_path_inspector(path; fidelity = 2.0,
                  title = "PathBuilt: three Subpaths (straight, bend, helix)")

<a id="sec3"></a>
## 3 · Modify — field-level MCM meta on segment fields

**Lesson of the class**: `MCMadd(:field, δ)` / `MCMmul(:field, f)` meta perturbs one
declared field of one segment, applied by `Fiber(builder; ...)` during its single
build. Plain `Float64` perturbations make the geometric meaning of each field
visible. With no anchor, everything downstream of the perturbed segment rides along
rigidly (contrast [§7](#sec7)). In every row: perturbed segment red, fixed segments
green, terminal connector faint gray.

In [ ]:
DEMO_XS = StepIndexCrossSection(
    SilicaGermaniaGlass(0.036),
    SilicaGermaniaGlass(0.0),
    8.2e-6,
    125e-6,
)
DEMO_T_K = 297.15

# Inverted-U baseline; `target_meta` attaches to segment `target_idx`
# (1 = first straight, 2 = π-bend, 3 = second straight).
function modify_variant(target_idx, target_meta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0, meta = target_idx == 1 ? target_meta : AbstractMeta[])
    bend!(sb; radius = 0.5, angle = π, axis_angle = 0.0,
          meta = target_idx == 2 ? target_meta : AbstractMeta[])
    straight!(sb; length = 1.0, meta = target_idx == 3 ? target_meta : AbstractMeta[])
    seal!(sb)
    return Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path
end

# Same with a helix inserted (1 = straight, 2 = bend, 3 = helix, 4 = straight).
function modify_variant_helix(target_idx, target_meta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0, meta = target_idx == 1 ? target_meta : AbstractMeta[])
    bend!(sb; radius = 0.5, angle = π, axis_angle = 0.0,
          meta = target_idx == 2 ? target_meta : AbstractMeta[])
    helix!(sb; radius = 0.15, pitch = 0.25, turns = 1.5, axis_angle = 0.0,
           meta = target_idx == 3 ? target_meta : AbstractMeta[])
    straight!(sb; length = 1.0, meta = target_idx == 4 ? target_meta : AbstractMeta[])
    seal!(sb)
    return Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path
end
println("builders defined")

### 3.1 · `MCMadd` on the inverted-U — `:length`, `:radius`, `:angle`

In [ ]:
dh_variant_row([("baseline",              modify_variant(1, AbstractMeta[]), Int[]),
                ("MCMadd(:length, -0.4)", modify_variant(1, [MCMadd(:length, -0.4)]), [1])];
               spacing = 2.5, gray_terminal = true,
               title = "MCMadd(:length) on the first straight")

In [ ]:
dh_variant_row([("baseline",               modify_variant(2, AbstractMeta[]), Int[]),
                ("MCMadd(:radius, -0.25)", modify_variant(2, [MCMadd(:radius, -0.25)]), [2]),
                ("MCMadd(:radius, +0.50)", modify_variant(2, [MCMadd(:radius,  0.50)]), [2])];
               spacing = 2.5, gray_terminal = true,
               title = "MCMadd(:radius) on the bend")

In [ ]:
dh_variant_row([("baseline",             modify_variant(2, AbstractMeta[]), Int[]),
                ("MCMadd(:angle, -π/2)", modify_variant(2, [MCMadd(:angle, -π / 2)]), [2]),
                ("MCMadd(:angle, +π/4)", modify_variant(2, [MCMadd(:angle,  π / 4)]), [2]),
                ("MCMadd(:angle, +π)",   modify_variant(2, [MCMadd(:angle,  π)]),     [2])];
               spacing = 2.5, gray_terminal = true,
               title = "MCMadd(:angle) on the bend — +π closes the circle")

### 3.2 · `MCMmul` on the inverted-U

**Edge case**: `MCMmul(:length, -0.4)` flips the sign — the first straight walks
*backward* along the local tangent.

In [ ]:
dh_variant_row([("baseline",              modify_variant(1, AbstractMeta[]), Int[]),
                ("MCMmul(:length, -0.4)", modify_variant(1, [MCMmul(:length, -0.4)]), [1]),
                ("MCMmul(:length, +0.5)", modify_variant(1, [MCMmul(:length,  0.5)]), [1])];
               spacing = 2.5, gray_terminal = true,
               title = "MCMmul(:length) on the first straight — negative walks backward")

In [ ]:
dh_variant_row([("baseline",             modify_variant(2, AbstractMeta[]), Int[]),
                ("MCMmul(:radius, 0.5)", modify_variant(2, [MCMmul(:radius, 0.5)]), [2]),
                ("MCMmul(:radius, 2.0)", modify_variant(2, [MCMmul(:radius, 2.0)]), [2])];
               spacing = 2.5, gray_terminal = true,
               title = "MCMmul(:radius) on the bend")

In [ ]:
dh_variant_row([("baseline",             modify_variant(2, AbstractMeta[]), Int[]),
                ("MCMmul(:angle, 0.5)",  modify_variant(2, [MCMmul(:angle, 0.5)]),  [2]),
                ("MCMmul(:angle, 1.25)", modify_variant(2, [MCMmul(:angle, 1.25)]), [2])];
               spacing = 2.5, gray_terminal = true,
               title = "MCMmul(:angle) on the bend")

### 3.3 · Helix fields

**Lesson**: the three helix knobs differ geometrically — `:radius` fattens the coil
(pitch fixed), `:pitch` stretches it axially, `:turns` changes the winding count and
hence the exit phase and direction.

In [ ]:
dh_variant_row([("baseline",               modify_variant_helix(3, AbstractMeta[]), Int[]),
                ("MCMadd(:radius, -0.05)", modify_variant_helix(3, [MCMadd(:radius, -0.05)]), [3]),
                ("MCMadd(:radius, +0.10)", modify_variant_helix(3, [MCMadd(:radius,  0.10)]), [3])];
               spacing = 3.5, gray_terminal = true,
               title = "MCMadd(:radius) on the helix")

In [ ]:
dh_variant_row([("baseline",              modify_variant_helix(3, AbstractMeta[]), Int[]),
                ("MCMadd(:pitch, -0.10)", modify_variant_helix(3, [MCMadd(:pitch, -0.10)]), [3]),
                ("MCMadd(:pitch, +0.20)", modify_variant_helix(3, [MCMadd(:pitch,  0.20)]), [3])];
               spacing = 3.5, gray_terminal = true,
               title = "MCMadd(:pitch) on the helix")

In [ ]:
dh_variant_row([("baseline",             modify_variant_helix(3, AbstractMeta[]), Int[]),
                ("MCMadd(:turns, -0.5)", modify_variant_helix(3, [MCMadd(:turns, -0.5)]), [3]),
                ("MCMadd(:turns, +0.5)", modify_variant_helix(3, [MCMadd(:turns,  0.5)]), [3])];
               spacing = 3.5, gray_terminal = true,
               title = "MCMadd(:turns) on the helix")

In [ ]:
dh_variant_row([("baseline",             modify_variant_helix(3, AbstractMeta[]), Int[]),
                ("MCMmul(:radius, 0.5)", modify_variant_helix(3, [MCMmul(:radius, 0.5)]), [3]),
                ("MCMmul(:radius, 2.0)", modify_variant_helix(3, [MCMmul(:radius, 2.0)]), [3])];
               spacing = 3.5, gray_terminal = true,
               title = "MCMmul(:radius) on the helix")

In [ ]:
dh_variant_row([("baseline",            modify_variant_helix(3, AbstractMeta[]), Int[]),
                ("MCMmul(:pitch, 0.5)", modify_variant_helix(3, [MCMmul(:pitch, 0.5)]), [3]),
                ("MCMmul(:pitch, 2.0)", modify_variant_helix(3, [MCMmul(:pitch, 2.0)]), [3])];
               spacing = 3.5, gray_terminal = true,
               title = "MCMmul(:pitch) on the helix")

In [ ]:
dh_variant_row([("baseline",             modify_variant_helix(3, AbstractMeta[]), Int[]),
                ("MCMmul(:turns, 0.67)", modify_variant_helix(3, [MCMmul(:turns, 0.67)]), [3]),
                ("MCMmul(:turns, 1.5)",  modify_variant_helix(3, [MCMmul(:turns, 1.5)]),  [3])];
               spacing = 3.5, gray_terminal = true,
               title = "MCMmul(:turns) on the helix")

<a id="sec4"></a>
## 4 · Material spin

**Lesson**: `start!(sb; spin_rate = 2π)` sets material spin for the whole Subpath;
the geometry layer resolves it into path-coordinate spin carried independently of
the frame geometry. Scrub the slider: τ_spin reads 6.2832 rad/m everywhere,
∫τ_spin ds grows linearly, and the red spin arrow rotates in the normal–binormal
plane while T̂/N̂/B̂ follow the helix's geometric torsion.

In [ ]:
sb = SubpathBuilder(); start!(sb; spin_rate = 2π)
straight!(sb; length = 1.0, meta = [Nickname("lead-in")])
helix!(sb; radius = 0.5, pitch = 0.05, turns = 4.0, axis_angle = 0.0,
       meta = [Nickname("helix")])
straight!(sb; length = 1.0, meta = [Nickname("lead-out")])
seal!(sb)
path = build(sb)

dh_path_inspector(path; title = "Helix with material spin (2π rad/m)")

<a id="sec5"></a>
## 5 · Adaptive step-doubling diagnostic

**Lesson**: the production step controller concentrates effort where the generator
varies fastest. On the smooth noncommuting generator

$$K(s) = α\,i\,σ_x\cos(πs) + β\,i\,σ_z\sin(2πs), \qquad α = 1.2,\; β = 0.9,\; s ∈ [0, 2]$$

`collect_adaptive_steps` (a recording mirror of `propagate_interval!`; the solver is
untouched) logs every trial. Watch for: step growth where ‖K(s)‖ dips, the short
rejection cascades (✕) right after, and err/tol hugging the threshold from below
with no accepted step above it.

In [ ]:
using Bifrost.Plots: collect_adaptive_steps
using LinearAlgebra

SX = ComplexF64[0 1; 1 0]
SZ = ComplexF64[1 0; 0 -1]
α, β = 1.2, 0.9
K = s -> α * im * cos(π * s) .* SX + β * im * sin(2π * s) .* SZ

J0 = Matrix{ComplexF64}(I, 2, 2)
J, records = collect_adaptive_steps(K, 0.0, 2.0, J0; rtol = 1e-6, atol = 1e-9)

println("accepted: ", count(r -> r.accepted, records),
        "   rejected: ", count(r -> !r.accepted, records))

dh_adaptive_panels(records, s -> opnorm(K(s)), 0.0, 2.0;
    rtol = 1e-6, atol = 1e-9,
    components = [("α·cos(πs)  [σx coeff]", s -> α * cos(π * s),  "#1f77b4"),
                  ("β·sin(2πs) [σz coeff]", s -> β * sin(2π * s), "#ff7f0e")],
    title = "Adaptive step-doubling on a noncommuting K(s)")

<a id="sec6"></a>
## 6 · JumpBy / JumpTo — 2D sweeps

**Lesson of the class**: `jumpby!` is an *interior* segment whose `delta` and
outgoing `tangent` live in the **local frame**; `jumpto!` *seals* a Subpath with a
connector to a **global** `point` (optional global `incoming_tangent` /
`incoming_curvature`), and the next Subpath continues via `start!(sb, :inherit)`.
Each sweep varies one connector degree of freedom; connector red, context green;
x–z projection (all paths lie in y = 0).

### 6.1 · JumpBy: outgoing tangent (local frame)

In [ ]:
function jumpby_tangent_path(tangent)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    jumpby!(sb; delta = (0.4, 0.0, 0.4), tangent = tangent)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

dh_variant_row_2d(
    [("t = (+1,0,1)/√2", jumpby_tangent_path((1 / sqrt(2), 0.0, 1 / sqrt(2))),  [2]),
     ("t = (0,0,1)",     jumpby_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("t = (-1,0,1)/√2", jumpby_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.0, title = "JumpBy: outgoing tangent sweep (local frame)")

### 6.2 · JumpTo: target point (global frame)

In [ ]:
function jumpto_point_path(point)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = point)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

dh_variant_row_2d(
    [("x = $(x)", jumpto_point_path((x, 0.0, 1.5)), [2]) for x in (0.0, 0.3, 0.6, 1.0)];
    spacing = 2.5, title = "JumpTo(point = (x, 0, 1.5)): transverse sweep")

### 6.3 · JumpTo: incoming tangent (global frame)

In [ ]:
function jumpto_tangent_path(incoming_tangent)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = (0.5, 0.0, 1.5), incoming_tangent = incoming_tangent)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

dh_variant_row_2d(
    [("incoming = (+1,0,0)",    jumpto_tangent_path((1.0, 0.0, 0.0)),                  [2]),
     ("incoming = (0,0,+1)",    jumpto_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("incoming = (-1,0,1)/√2", jumpto_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.5, title = "JumpTo: incoming tangent sweep (global frame)")

### 6.4 · JumpTo routing: a serpentine of waypoints

**Lesson**: composite routing through global waypoints with anti-parallel landing
tangents; each `jumpto!` seals a Subpath, so the chain is a 3-Subpath `PathBuilt`.

In [ ]:
sb1 = SubpathBuilder(); start!(sb1)
straight!(sb1; length = 1.0)
jumpto!(sb1; point = (1.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.1)
sb2 = SubpathBuilder(); start!(sb2, :inherit)
straight!(sb2; length = 1.0)
jumpto!(sb2; point = (2.0, 0.0, 0.0), incoming_tangent = (0.0, 0.0, 1.0),
        min_bend_radius = 0.1)
sb3 = SubpathBuilder(); start!(sb3, :inherit)
straight!(sb3; length = 1.0)
jumpto!(sb3; point = (3.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.1)
routing = build([sb1, sb2, sb3])

dh_variant_row_2d([("composite", routing, [2, 4, 6])];
                  spacing = 0.0, title = "JumpTo routing: serpentine through waypoints")

### 6.5 · JumpTo `min_bend_radius`: a feasibility sweep

**Lesson**: `min_bend_radius` is a hard build-time guardrail — infeasible variants
throw at `build` (trapped here; rendered as the surviving lead-in, labeled
*(infeasible)*). **Observed**: for this U-turn over a transverse unit chord, the
threshold lies between 0.30 and 0.49 — below the circular-arc ideal chord/2 = 0.5,
because the quintic connector's peak curvature necessarily exceeds the circular
bound.

In [ ]:
function mbr_path(mbr)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = (1.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
            min_bend_radius = mbr)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

lead_in_only() = begin
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    seal!(sb)
    build(sb)
end

variants = map((0.10, 0.30, 0.49, 0.51, 0.70)) do mbr
    path, err = dh_try_build(() -> mbr_path(mbr))
    err === nothing ? ("mbr = $(mbr)", path, [2]) :
                      ("mbr = $(mbr) (infeasible)", lead_in_only(), Int[])
end

dh_variant_row_2d(collect(variants); spacing = 2.0,
    title = "JumpTo min_bend_radius sweep — U-turn over a transverse unit chord")

<a id="sec7"></a>
## 7 · Meta × JumpTo interplay — anchored vs unanchored perturbations

**Lesson of the class**: where do upstream perturbations go? (1) with `jumpby!`
there is no anchor — the endpoint drifts; (2) a `jumpto!` anchor pins the endpoint
while its connector absorbs the slack; (3) `:T_K` on the `jumpto!` seal expands the
connector itself without moving the anchor. Overlays: baseline black, modified red,
lengths and Δ% in the legend.

### 7.1 · S-curve + JumpBy: downstream drifts

Salient: `MCMmul(:radius, 1.25)` on both bends; the `jumpby!` delta is
fiber-relative, so the whole tail shifts (~10% length growth, endpoint separates).

In [ ]:
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π)
straight!(sb; length = 0.3)
jumpby!(sb; delta = (0.0, 0.0, 0.8))
straight!(sb; length = 1.0)
seal!(sb)
baseline = build(sb)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0, meta = [MCMmul(:radius, 1.25)])
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π,   meta = [MCMmul(:radius, 1.25)])
straight!(sb; length = 0.3)
jumpby!(sb; delta = (0.0, 0.0, 0.8))
straight!(sb; length = 1.0)
seal!(sb)
modified = Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path

dh_overlay_compare(baseline, modified; legend = :bottomright,
    title = "S-curve + JumpBy: no anchor — downstream drifts")

### 7.2 · S-curve + JumpTo: the anchor pins the endpoint

Salient: same S-curve sealed by `jumpto!` to a fixed waypoint (placed so the
baseline totals 4.000 m); a stronger `MCMmul(:radius, 1.5)` swings the interior
wide, yet the endpoint stays pinned — the connector absorbs the slack.

In [ ]:
probe = SubpathBuilder(); start!(probe)
straight!(probe; length = 0.3)
bend!(probe; radius = 0.5, angle = π / 2, axis_angle = 0.0)
bend!(probe; radius = 0.5, angle = π / 2, axis_angle = π)
straight!(probe; length = 0.3)
seal!(probe)
pre = build(probe)
p_end = end_point(pre)
anchor = (p_end[1], p_end[2], p_end[3] + (4.0 - path_length(pre)))

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π)
straight!(sb; length = 0.3)
jumpto!(sb; point = anchor)
baseline = build(sb)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0, meta = [MCMmul(:radius, 1.5)])
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π,   meta = [MCMmul(:radius, 1.5)])
straight!(sb; length = 0.3)
jumpto!(sb; point = anchor)
modified = Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path

dh_overlay_compare(baseline, modified;
    title = "S-curve + JumpTo: anchor pins the endpoint, connector absorbs slack")

### 7.3 · `:T_K` on the JumpTo seal: the connector itself expands

Salient: `MCMadd(:T_K, ΔT)` on **both** the straight and its sealing `jumpto!`
(ΔT sized for 5% expansion via the cladding CTE). The connector's arc length scales
by τ while still landing on the fixed point — extra length becomes connector
curvature. `:T_K` is interpreted by `Fiber`, never by the geometry layer.

In [ ]:
α_lin = cte(DEMO_XS.cladding_material, DEMO_T_K)
ΔT  = 0.05 / α_lin
mdt = [MCMadd(:T_K, ΔT)]

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5)
jumpto!(sb; point = (1.0, 0.0, 0.5), incoming_tangent = (1.0, 0.0, 0.0))
baseline = build(sb)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5, meta = mdt)
jumpto!(sb; point = (1.0, 0.0, 0.5), incoming_tangent = (1.0, 0.0, 0.0), meta = mdt)
modified = Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path

dh_overlay_compare(baseline, modified;
    title = ":T_K on segment + JumpTo seal: connector thermally expands, anchor holds")

<a id="sec8"></a>
## 8 · JumpBy / JumpTo — 3D scenes

The [§6](#sec6) lessons in 3D plus the G2-continuity content that needs the third
dimension. Connector red, fixed segments green, variants offset along +x.

### 8.1 · JumpBy delta sweeps (local frame)

In [ ]:
function jumpby_delta_path(delta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    jumpby!(sb; delta = delta)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

dh_variant_row([("d = $(d)", jumpby_delta_path((0.0, 0.0, d)), [2]) for d in (0.3, 0.6, 1.0)];
               spacing = 2.5, title = "JumpBy(delta = (0, 0, d)) — axial sweep")

In [ ]:
dh_variant_row([("d = $(d)", jumpby_delta_path((d, 0.0, 0.5)), [2]) for d in (0.0, 0.2, 0.5)];
               spacing = 2.5, title = "JumpBy(delta = (d, 0, 0.5)) — transverse sweep")

### 8.2 · JumpBy: outgoing tangent and outgoing curvature

**Lesson**: `curvature_out` is the G2 exit knob — the (0, ±2, 0) variants leave the
connector bowing out of the x–z plane.

In [ ]:
dh_variant_row(
    [("t = (+1,0,1)/√2", jumpby_tangent_path((1 / sqrt(2), 0.0, 1 / sqrt(2))),  [2]),
     ("t = (0,0,1)",     jumpby_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("t = (-1,0,1)/√2", jumpby_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.5, title = "JumpBy — outgoing tangent (local frame)")

In [ ]:
function jumpby_curvature_path(curvature_out)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    jumpby!(sb; delta = (0.5, 0.0, 0.5), tangent = (1.0, 0.0, 1.0) ./ sqrt(2),
            curvature_out = curvature_out)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

dh_variant_row(
    [("κ_out = 0",        jumpby_curvature_path((0.0, 0.0, 0.0)),  [2]),
     ("κ_out = (0,+2,0)", jumpby_curvature_path((0.0, 2.0, 0.0)),  [2]),
     ("κ_out = (0,-2,0)", jumpby_curvature_path((0.0, -2.0, 0.0)), [2])];
    spacing = 2.5, title = "JumpBy — outgoing curvature (G2 exit knob)")

### 8.3 · JumpTo point and landing sweeps (global frame)

In [ ]:
dh_variant_row([("z = $(z)", jumpto_point_path((0.0, 0.0, z)), [2]) for z in (1.3, 1.6, 2.0)];
               spacing = 2.5, title = "JumpTo(point = (0, 0, z)) — axial sweep")

In [ ]:
dh_variant_row([("x = $(x)", jumpto_point_path((x, 0.0, 1.5)), [2]) for x in (0.0, 0.3, 0.6)];
               spacing = 2.5, title = "JumpTo(point = (x, 0, 1.5)) — transverse sweep")

In [ ]:
dh_variant_row(
    [("incoming = (+1,0,0)",    jumpto_tangent_path((1.0, 0.0, 0.0)),                  [2]),
     ("incoming = (0,0,+1)",    jumpto_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("incoming = (-1,0,1)/√2", jumpto_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.5, title = "JumpTo — incoming tangent (global frame)")

In [ ]:
function jumpto_curvature_path(incoming_curvature)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = (0.5, 0.0, 1.5), incoming_tangent = (0.0, 0.0, 1.0),
            incoming_curvature = incoming_curvature)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

dh_variant_row(
    [("κ_in = 0",         jumpto_curvature_path((0.0, 0.0, 0.0)),   [2]),
     ("κ_in = (+10,0,0)", jumpto_curvature_path((10.0, 0.0, 0.0)),  [2]),
     ("κ_in = (-10,0,0)", jumpto_curvature_path((-10.0, 0.0, 0.0)), [2])];
    spacing = 1.5, title = "JumpTo — incoming curvature (G2 landing knob)")

### 8.4 · After a bend: rotated local frame vs global target

**Lesson**: after a 90° bend, a `jumpby!` delta is expressed in the **rotated**
local frame; a `jumpto!` target stays **global** and does not rotate with the path.

In [ ]:
function jumpby_after_bend_path(delta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0)
    jumpby!(sb; delta = delta)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

dh_variant_row(
    [("delta = (0,0,0.5)",    jumpby_after_bend_path((0.0, 0.0, 0.5)),  [3]),
     ("delta = (0.3,0,0.5)",  jumpby_after_bend_path((0.3, 0.0, 0.5)),  [3]),
     ("delta = (-0.3,0,0.5)", jumpby_after_bend_path((-0.3, 0.0, 0.5)), [3])];
    spacing = 3.5, title = "JumpBy after 90° bend — delta in the ROTATED local frame")

In [ ]:
function jumpto_after_bend_path(point)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    bend!(sb1; radius = 0.5, angle = π / 2, axis_angle = 0.0)
    jumpto!(sb1; point = point)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

dh_variant_row(
    [("dest = (1.0, 0, 1.5)", jumpto_after_bend_path((1.0, 0.0, 1.5)), [3]),
     ("dest = (1.3, 0, 1.5)", jumpto_after_bend_path((1.3, 0.0, 1.5)), [3]),
     ("dest = (0.5, 0, 2.0)", jumpto_after_bend_path((0.5, 0.0, 2.0)), [3])];
    spacing = 3.5, title = "JumpTo after 90° bend — point in the GLOBAL frame")

### 8.5 · G2 inheritance from the preceding segment

**Lesson**: with no explicit curvature spec, a connector inherits the preceding
bend's exit curvature κ = 1/R (G2 continuity by default) — smaller R launches the
connector on a visibly tighter initial arc.

In [ ]:
function g2_inheritance_path(R_bend)
    sb = SubpathBuilder(); start!(sb)
    bend!(sb; radius = R_bend, angle = π / 4, axis_angle = 0.0)
    jumpby!(sb; delta = (0.3, 0.0, 0.3))
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

dh_variant_row(
    [("R = 0.30 (κ_in ≈ 3.33)", g2_inheritance_path(0.30), [2]),
     ("R = 0.50 (κ_in = 2.00)", g2_inheritance_path(0.50), [2]),
     ("R = 0.80 (κ_in = 1.25)", g2_inheritance_path(0.80), [2])];
    spacing = 3.0, title = "JumpBy after a bend — G2 inheritance of incoming κ")

### 8.6 · Routing in 3D

In [ ]:
dh_variant_row([("composite", routing, [2, 4, 6])];
               spacing = 0.0, title = "JumpTo routing — serpentine (3D view)")

<a id="sec9"></a>
## 9 · MCM temperature PTF — end-to-end uncertainty

**Lesson of the class**: a full MCM pipeline — uncertain temperature entering twice
(as `MCMadd(:T_K, ΔT)` geometry meta on the first helix and as `T_ref_K` on the
final `Fiber` binding for material indices), one ensemble propagation, per-particle
Stokes post-processing.

**Physics**: bend birefringence scales as 1/R²; at R = 2.5 cm the ~10000-turn helix
accumulates |Δβ|·L ≈ 1775·2π rad. The fractional turn count (10001.892…) puts the
30 °C operating point exactly at **mid-fringe** (mod(Γ, 2π) = π), where dPTF/dT is
maximal (at a crossing it is zero). Over ±5 °C the retardation swings ≈ ±0.75π. The
output stays nearly linearly polarized (DLP ≈ 1, S3 ≈ 0): the sensitive observable
is the polarization **angle**.

In [ ]:
using Distributions

MCM_DEMO_XS = StepIndexCrossSection(
    SilicaGermaniaGlass(0.036),
    SilicaGermaniaGlass(0.0),
    8.2e-6,
    125e-6,
)
MCM_DEMO_T_REF_K = 303.15   # 30 °C reference temperature (= design point)
MCM_DEMO_λ_M     = 1550e-9
MCM_DEMO_T_NOM_C = 30.0     # nominal operating temperature (°C)
MCM_DEMO_T_SIG_C = 5.0      # ±5 °C uncertainty (1σ)

# turns chosen so Δβ(30 °C)·L = odd multiple of π (mid-fringe):
MCM_DEMO_HELIX1_TURNS = 10001.892069208387

# Build the 5-segment fiber spec. `ΔT_K` is the temperature offset applied to
# the first helix via MCMadd(:T_K, ΔT_K); pass 0.0 for the baseline.
function mcm_demo_fiber(ΔT_K)
    # Sinusoidal spin over the whole Subpath: amplitude 1 rad/m, period 100 m.
    sb = SubpathBuilder(); start!(sb; spin_rate = s -> sin(2π * s / 100.0))
    straight!(sb; length = 5.0)
    helix!(sb; radius = 0.025, pitch = 0.05, turns = MCM_DEMO_HELIX1_TURNS,
           axis_angle = 0.0,
           meta = AbstractMeta[Nickname("temperature-sensitive helix"),
                               MCMadd(:T_K, ΔT_K)])
    straight!(sb; length = 5.0, meta = [Nickname("spacer")])
    helix!(sb; radius = 0.025, pitch = 0.05, turns = 10000.0,
           axis_angle = 0.0, meta = [Nickname("reference helix")])
    straight!(sb; length = 5.0, meta = [Nickname("lead-out")])
    seal!(sb)
    return Fiber(sb; cross_section = MCM_DEMO_XS, T_ref_K = MCM_DEMO_T_REF_K)
end
println("mcm_demo_fiber defined")

### 9.1 · StaticParticles(50): angle and Stokes vs temperature

In [ ]:
Random.seed!(20260612)
MonteCarloMeasurements.unsafe_comparisons(true)
T_C_particles  = StaticParticles(50, Normal(MCM_DEMO_T_NOM_C, MCM_DEMO_T_SIG_C))
T_K_particles  = T_C_particles + 273.15
ΔT_K_particles = T_K_particles - MCM_DEMO_T_REF_K

fiber_mcm     = mcm_demo_fiber(ΔT_K_particles)
modified_path = fiber_mcm.path  # thermal already applied at construction
fiber_mod     = Fiber(modified_path; cross_section = MCM_DEMO_XS,
                      T_ref_K = T_K_particles)
J_p, _        = propagate_fiber(fiber_mod; λ_m = MCM_DEMO_λ_M)
MonteCarloMeasurements.unsafe_comparisons(false)

st = dh_stokes_ensemble(J_p)   # per-particle Jones → Stokes, input state H
dh_temperature_ptf_row(T_C_particles.particles, st;
    title = "MCM temperature PTF — StaticParticles(50), T = 30 °C ± 5 °C, λ = 1550 nm")

### 9.2 · Particles(500): ensemble scatter on the Poincaré equator

One heap-backed `Particles` run; the ensemble traces a temperature-parameterized arc
on the S1–S2 equatorial projection. *(500 particles; raise N for denser arcs on
faster hardware.)*

In [ ]:
Random.seed!(20260612)
MonteCarloMeasurements.unsafe_comparisons(true)
T_C_particles  = Particles(500, Normal(MCM_DEMO_T_NOM_C, MCM_DEMO_T_SIG_C))
T_K_particles  = T_C_particles + 273.15
ΔT_K_particles = T_K_particles - MCM_DEMO_T_REF_K

fiber_mcm     = mcm_demo_fiber(ΔT_K_particles)
fiber_mod     = Fiber(fiber_mcm.path; cross_section = MCM_DEMO_XS,
                      T_ref_K = T_K_particles)
J_p, _        = propagate_fiber(fiber_mod; λ_m = MCM_DEMO_λ_M)
MonteCarloMeasurements.unsafe_comparisons(false)

st = dh_stokes_ensemble(J_p)
dh_temperature_scatter_row(T_C_particles.particles, st;
    title = "MCM temperature PTF scatter — Particles(500), T = 30 °C ± 5 °C")

<a id="sec10"></a>
## 10 · MCM speed benchmarks

**Lesson**: the wall-time cost of ensemble propagation vs a nominal `Float64` run,
and the `StaticParticles` SIMD sweet spot at small N. Per case: **first-call**
(JIT + one run) and **steady-state** (BenchmarkTools minimum over 3 post-JIT
samples). Numbers are machine-dependent — nothing is asserted.

*Scaling note: same five-segment structure as [§9](#sec9) but with 100-turn helices
(the legacy fiber's mid-fringe tuning matters for sensitivity, not speed) so the
repeated timed runs — in particular Particles(2000) — stay tractable. Raise `turns`
in `bench_build_fiber` on faster hardware.*

In [ ]:
using BenchmarkTools

# Construct an MCM-annotated fiber ready for propagate_fiber. `make_T` returns a
# temperature value (Float64, Particles, or StaticParticles) at 30 °C.
function bench_build_fiber(make_T)
    T_val = make_T()
    T_K   = T_val + 273.15
    ΔT_K  = T_K - MCM_DEMO_T_REF_K
    sb = SubpathBuilder(); start!(sb; spin_rate = s -> sin(2π * s / 100.0))
    straight!(sb; length = 5.0)
    helix!(sb; radius = 0.025, pitch = 0.05, turns = 100.0, axis_angle = 0.0,
           meta = AbstractMeta[MCMadd(:T_K, ΔT_K)])
    straight!(sb; length = 5.0)
    helix!(sb; radius = 0.025, pitch = 0.05, turns = 100.0, axis_angle = 0.0)
    straight!(sb; length = 5.0)
    seal!(sb)
    fiber = Fiber(sb; cross_section = MCM_DEMO_XS, T_ref_K = MCM_DEMO_T_REF_K)
    return Fiber(fiber.path; cross_section = MCM_DEMO_XS, T_ref_K = T_K)
end

# Time the first call to f() (JIT + one run) in milliseconds.
first_call_ms(f) = (t0 = time_ns(); f(); (time_ns() - t0) / 1e6)

bench_cases = [
    ("Float64",             0,    () -> MCM_DEMO_T_NOM_C),
    ("Particles(2000)",     2000, () -> Particles(2000, Normal(MCM_DEMO_T_NOM_C, MCM_DEMO_T_SIG_C))),
    ("StaticParticles(50)", 50,   () -> StaticParticles(50, Normal(MCM_DEMO_T_NOM_C, MCM_DEMO_T_SIG_C))),
    ("StaticParticles(75)", 75,   () -> StaticParticles(75, Normal(MCM_DEMO_T_NOM_C, MCM_DEMO_T_SIG_C))),
]
println("benchmark scaffolding defined")

### 10.1 · `propagate_fiber` alone (fiber pre-built)

In [ ]:
Random.seed!(20260612)
MonteCarloMeasurements.unsafe_comparisons(true)
results = map(bench_cases) do (label, n_p, make_T)
    fiber = bench_build_fiber(make_T)
    fn() = propagate_fiber(fiber; λ_m = MCM_DEMO_λ_M, verbose = false)
    first  = first_call_ms(fn)
    steady = @belapsed($fn(); samples = 3, evals = 1) * 1e3
    println(rpad(label, 22), "first ", lpad(round(first; digits = 1), 10), " ms",
            "   steady ", lpad(round(steady; digits = 1), 10), " ms")
    (; label, n_particles = n_p, first_ms = first, steady_ms = steady)
end
MonteCarloMeasurements.unsafe_comparisons(false)

display(dh_benchmark_chart(collect(results);
    title = "propagate_fiber — first-call vs steady-state (log scale)"))
dh_benchmark_table(collect(results))

### 10.2 · Build + propagate (the full per-sample pipeline)

Timing includes the `:T_K` thermal geometry build, `Fiber` construction, and
propagation.

In [ ]:
Random.seed!(20260612)
MonteCarloMeasurements.unsafe_comparisons(true)
results = map(bench_cases) do (label, n_p, make_T)
    fn() = propagate_fiber(bench_build_fiber(make_T); λ_m = MCM_DEMO_λ_M,
                           verbose = false)
    first  = first_call_ms(fn)
    steady = @belapsed($fn(); samples = 3, evals = 1) * 1e3
    println(rpad(label, 22), "first ", lpad(round(first; digits = 1), 10), " ms",
            "   steady ", lpad(round(steady; digits = 1), 10), " ms")
    (; label, n_particles = n_p, first_ms = first, steady_ms = steady)
end
MonteCarloMeasurements.unsafe_comparisons(false)

display(dh_benchmark_chart(collect(results);
    title = "build + propagate_fiber — first-call vs steady-state (log scale)"))
dh_benchmark_table(collect(results))